The goal of this notebook is to wrangle the data required as input for the atlas comparison plot for Figure 2.

In [1]:
import os 
from pathlib import Path
import warnings 

from dotenv import dotenv_values
import pandas as pd
import numpy as np

warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
# Setting up filepaths from config
CONFIG = dotenv_values("config.env") 
DATADIR = Path(CONFIG['DATADIR'])

OUTDIR = DATADIR / "publication" / "Fig_02"
os.makedirs(OUTDIR, exist_ok=True)

CLUSTERING_COMP_DIR = DATADIR / "10_analyses" / "clustering_comparison"
REGION_COMP_DIR = DATADIR / "10_analyses" / "region_comparison"

# Preparation of input data to the plotting function

### Let's load the data

In [3]:
all_overlaps_l2 = pd.read_csv(REGION_COMP_DIR / "region_prediction_comparison_to_other_atlases_L2.csv")
all_overlaps_l3 = pd.read_csv(REGION_COMP_DIR / "region_prediction_comparison_to_other_atlases_L3.csv")

abc_inheritance_l2 = pd.read_csv(CLUSTERING_COMP_DIR / "inheritance_status_l2.csv") 
abc_inheritance_l3 = pd.read_csv(CLUSTERING_COMP_DIR / "inheritance_status_l3.csv")
hmv1_inheritance_l2 = pd.read_csv(CLUSTERING_COMP_DIR / "inheritance_status_l2_hmv1.csv")
hmv1_inheritance_l3 = pd.read_csv(CLUSTERING_COMP_DIR / "inheritance_status_l3_hmv1.csv") 

### Time to wrangle

The overlap dfs from the region analyses still have to be corrected for the hierarchy. 

This can be done using the inheritance status from the inheritance/clustering comparison analysis. 

In [4]:
# ensure clusters are annotated as ID 
all_overlaps_l2.hm3d_cluster = all_overlaps_l2.hm3d_cluster.apply(lambda x: x.split(" ")[0])
all_overlaps_l3.hm3d_cluster = all_overlaps_l3.hm3d_cluster.apply(lambda x: x.split(" ")[0])

abc_inheritance_l2.source_label = abc_inheritance_l2.source_label.apply(lambda x: x.split(" ")[0])
abc_inheritance_l3.source_label_subclass = abc_inheritance_l3.source_label_subclass.apply(lambda x: x.split(" ")[0])
abc_inheritance_l3.source_label_supertype = abc_inheritance_l3.source_label_supertype.apply(lambda x: x.split(" ")[0])

hmv1_inheritance_l2.source_label = hmv1_inheritance_l2.source_label.apply(lambda x: x.split(" ")[0])
hmv1_inheritance_l3.source_label_subclass = hmv1_inheritance_l3.source_label_subclass.apply(lambda x: x.split(" ")[0])
hmv1_inheritance_l3.source_label_supertype = hmv1_inheritance_l3.source_label_supertype.apply(lambda x: x.split(" ")[0])

In [5]:
# let's join the inheritance and the most probable match match % 

##### L2

all_overlaps_l2 = all_overlaps_l2.set_index("hm3d_cluster")

hmv1_inheritance_l2 = hmv1_inheritance_l2.set_index("source_label")
abc_inheritance_l2 = abc_inheritance_l2.set_index("source_label")

all_overlaps_l2["hmv1_inheritance_status"] = hmv1_inheritance_l2.inheritance_status.copy()
all_overlaps_l2["hmv1_most_probable_match_counts_relative"] = hmv1_inheritance_l2.most_probable_match_counts_relative.copy()
all_overlaps_l2["abc_inheritance_status"] = abc_inheritance_l2.inheritance_status.copy()
all_overlaps_l2["abc_most_probable_match_counts_relative"] = abc_inheritance_l2.most_probable_match_counts_relative.copy()


##### L3

all_overlaps_l3 = all_overlaps_l3.set_index("hm3d_cluster")

hmv1_inheritance_l3 = hmv1_inheritance_l3.set_index("source_label_supertype")
abc_inheritance_l3 = abc_inheritance_l3.set_index("source_label_supertype")

all_overlaps_l3["hmv1_mapping_category"] = hmv1_inheritance_l3.mapping_category.copy()
all_overlaps_l3["hmv1_most_probable_match_counts_relative"] = hmv1_inheritance_l3.most_probable_match_counts_relative_supertype.copy()
all_overlaps_l3["abc_mapping_category"] = abc_inheritance_l3.mapping_category.copy()
all_overlaps_l3["abc_most_probable_match_counts_relative"] = abc_inheritance_l3.most_probable_match_counts_relative_supertype.copy()


For the overlap dfs, all matches were considered, regardless of whether they met the 0.66 threshold.

Hence, overlap results for those clusters must be overwritten with "no match". These clusters are defined by being flagged as "new" clusters in the inheritance while having rightfully less than 0.66 as relative match count. 

On the other hand, if clusters were flagged as new clusters in the inheritance while having relative match counts > 0.66 this must imply that the cluster had been violating the hierarchy. Of course, this can only occur at L3 as we started matching at L2. Therefore, the hierarchy can only be violated starting at L3.



In [6]:

for atlas in ["hmv1", "abc"]:

    ######### L2

    # overwrite all columns where there are new clusters 
    mask_new = (all_overlaps_l2[f"{atlas}_inheritance_status"] == "new cluster") & (
      all_overlaps_l2[f"{atlas}_most_probable_match_counts_relative"] <= 0.66)

    all_overlaps_l2.loc[mask_new, f"overlap_percent_{atlas}"] = "no match"
    all_overlaps_l2.loc[mask_new, f"matched_cluster_{atlas}"] = "no match"
    all_overlaps_l2.loc[mask_new, f"top_3_nuclei_other_{atlas}"] = "no match"
    all_overlaps_l2.loc[mask_new, f"top_3_nuclei_intersection_{atlas}"] = "no match"
    all_overlaps_l2.loc[mask_new, f"hm3d_to_{atlas}_match_category"] = "no cluster match"

    # # sanity check for hierarchy violation at L2 which cannot exist
    mask_violates_hierarchy = (all_overlaps_l2[f"{atlas}_inheritance_status"] == "new cluster") & (
        all_overlaps_l2[f"{atlas}_most_probable_match_counts_relative"] > 0.66)
    assert mask_violates_hierarchy.sum() == 0, "there are new subclass with >0.66 but at L2 there cannot be any hierarchy violations yet"

    ######### L3

      # overwrite all columns where there are new clusters 
    mask_new = (all_overlaps_l3[f"{atlas}_mapping_category"] == "new") & (
       all_overlaps_l3[f"{atlas}_most_probable_match_counts_relative"] <= 0.66)

    all_overlaps_l3.loc[mask_new, f"overlap_percent_{atlas}"] = "no match"
    all_overlaps_l3.loc[mask_new, f"matched_cluster_{atlas}"] = "no match"
    all_overlaps_l3.loc[mask_new, f"top_3_nuclei_other_{atlas}"] = "no match"
    all_overlaps_l3.loc[mask_new, f"top_3_nuclei_intersection_{atlas}"] = "no match"
    all_overlaps_l3.loc[mask_new, f"hm3d_to_{atlas}_match_category"] = "no cluster match"

    # set new category for hierarchy violation
    mask_violates_hierarchy = (all_overlaps_l3[f"{atlas}_mapping_category"] == "new") & (
        all_overlaps_l3[f"{atlas}_most_probable_match_counts_relative"] > 0.66)

    all_overlaps_l3.loc[mask_violates_hierarchy, f"{atlas}_mapping_category"] = "match violates hierarchy"

### Let's compile the data 

In [7]:
compiled = pd.concat([
    all_overlaps_l2[[
        "overlap_percent_abc",
        "hm3d_to_abc_match_category",
        "abc_inheritance_status",
        "overlap_percent_hmv1",
        "hm3d_to_hmv1_match_category",
        "hmv1_inheritance_status",
        ]].copy().rename(columns={
            "hmv1_inheritance_status": "hmv1_mapping_category",
            "abc_inheritance_status": "abc_mapping_category"
        }),

    all_overlaps_l3[[
        "overlap_percent_abc",
        "hm3d_to_abc_match_category",
        "abc_mapping_category",
        "overlap_percent_hmv1",
        "hm3d_to_hmv1_match_category",
        "hmv1_mapping_category",
        ]].copy(),
])



In [8]:
compiled = compiled.rename(columns={
    "hm3d_to_abc_match_category": "spatial_top_nuclei_match_category_abc",
    "abc_mapping_category": "cluster_mapping_category_abc",
    "hm3d_to_hmv1_match_category": "spatial_top_nuclei_match_category_hmv1",
    "hmv1_mapping_category": "cluster_mapping_category_hmv1",
    "overlap_percent_abc": "spatial_pred_overlap_percent_abc",
    "overlap_percent_hmv1":  "spatial_pred_overlap_percent_hmv1"
})


In [9]:
compiled.cluster_mapping_category_abc = compiled.cluster_mapping_category_abc.replace({
    "new cluster": "new",
    "inherited": "1:1", 
    "inherited and deduplicated": "many:1",
    'new (violates hierarchy)': "match violates hierarchy"
    })

compiled.cluster_mapping_category_hmv1 = compiled.cluster_mapping_category_hmv1.replace({
    "new cluster": "new",
    "inherited": "1:1", 
    "inherited and deduplicated": "many:1",
    'new (violates hierarchy)': "match violates hierarchy"
    })

compiled.spatial_pred_overlap_percent_abc = compiled.spatial_pred_overlap_percent_abc.replace({
    "no match": "no cluster match",
    })

compiled.spatial_pred_overlap_percent_hmv1 = compiled.spatial_pred_overlap_percent_hmv1.replace({
    "no match": "no cluster match",
    })

In [10]:
compiled = compiled.reset_index()
compiled["level"] = compiled.hm3d_cluster.apply(lambda x: f"L{x[2]}")

In [11]:

# Melt all atlas-specific columns
df_melted = pd.melt(
    compiled,
    id_vars=['hm3d_cluster', 'level'],  # these stay as-is
    value_vars=[
        'spatial_pred_overlap_percent_abc',
        'spatial_top_nuclei_match_category_abc',
        'cluster_mapping_category_abc',
        'spatial_pred_overlap_percent_hmv1',
        'spatial_top_nuclei_match_category_hmv1',
        'cluster_mapping_category_hmv1',
    ],
    var_name='analysis_atlas',
    value_name='value'
)

# Split 'analysis_atlas' into two columns: 'analysis' and 'atlas'
df_melted[['analysis', 'atlas']] = df_melted['analysis_atlas'].str.rsplit('_', n=1, expand=True)

# Drop the original combined column
df_melted = df_melted.drop(columns='analysis_atlas')

# Optional: rearrange columns for readability
df_long = df_melted[['hm3d_cluster', 'level', 'atlas', 'analysis', 'value']]

df_long.value = df_long.value.replace({"new": "no cluster match"})

In [12]:
df_long

,hm3d_cluster,level,atlas,analysis,value
0,Gb2-39,L2,abc,spatial_pred_overlap_percent,0.531298
1,Gb2-40,L2,abc,spatial_pred_overlap_percent,0.127419
2,Gb2-41,L2,abc,spatial_pred_overlap_percent,0.659388
3,Gb2-42,L2,abc,spatial_pred_overlap_percent,0.524818
4,Gb2-43,L2,abc,spatial_pred_overlap_percent,0.628486
...,...,...,...,...,...
1879,Gl3-119,L3,hmv1,cluster_mapping_category,many:1
1880,Gl3-120,L3,hmv1,cluster_mapping_category,no cluster match
1881,Gl3-121,L3,hmv1,cluster_mapping_category,many:1
1882,Gl3-122,L3,hmv1,cluster_mapping_category,1:1


### Let's save 

In [13]:
df_long.to_csv(OUTDIR / "region_and_cluster_comparison_long_df_for_plot.csv", index=False)